In [63]:
import torch
import torch.nn as nn 
from torch.nn import functional as F 
torch.manual_seed(20202)

In [64]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False,      # Query-Key-Value bias
    "n_experts":12          # No of Experts
} 

In [65]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [66]:
class Expert(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(GPT_CONFIG_124M["emb_dim"],4*GPT_CONFIG_124M["emb_dim"]),
            GELU(),
            nn.Linear(4*GPT_CONFIG_124M["emb_dim"],GPT_CONFIG_124M["emb_dim"]),
            nn.Dropout(GPT_CONFIG_124M['drop_rate'])
        )
    def forward(self,x):
        return self.net(x)

In [113]:
class Router(nn.Module):
    def __init__(self,topk=3):
        super().__init__()
        self.top_k=topk
        self.router=nn.Linear(GPT_CONFIG_124M["emb_dim"],GPT_CONFIG_124M["n_experts"])
    def forward(self,x):
        expert_selector=self.router(x)
        topk_val,indices=torch.topk(expert_selector,k=self.top_k,dim=-1)
        mask=torch.full_like(expert_selector,float('-inf'))
        expert_selector=mask.scatter(-1,indices,topk_val)
        gating_output=torch.softmax(expert_selector,dim=-1)
        return gating_output,indices
        


### Tensor operation practice

In [68]:
tensor=torch.randn((4,8))

In [84]:
tensor

tensor([[ 1.9125,  0.0299,  1.9976, -2.8078, -2.0682, -0.0959,  0.3155, -0.6504],
        [-1.1580, -0.7878, -0.5336,  0.8184, -0.7436, -0.2318, -0.0496, -0.1987],
        [ 1.1037, -0.8186,  0.4222,  1.4338,  0.4561, -1.3989, -2.2847,  1.0232],
        [-0.0937,  0.1944,  0.4714,  0.2684, -0.9570, -1.2337, -0.0324, -0.3018]])

In [ ]:
value,index=tensor.topk(3)

torch.return_types.topk(
values=tensor([[ 1.9976,  1.9125,  0.3155],
        [ 0.8184, -0.0496, -0.1987],
        [ 1.4338,  1.1037,  1.0232],
        [ 0.4714,  0.2684,  0.1944]]),
indices=tensor([[2, 0, 6],
        [3, 6, 7],
        [3, 0, 7],
        [2, 3, 1]]))

In [79]:
index

tensor([[ 1.9976,  1.9125,  0.3155],
        [ 0.8184, -0.0496, -0.1987],
        [ 1.4338,  1.1037,  1.0232],
        [ 0.4714,  0.2684,  0.1944]])

In [111]:
mask=mask.scatter(-1,value,index) ## -1 being last dim, always on same row 
torch.softmax(mask,dim=-1)

tensor([[0.4364, 0.0000, 0.4752, 0.0000, 0.0000, 0.0000, 0.0884, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.5613, 0.0000, 0.0000, 0.2356, 0.2030],
        [0.3018, 0.0000, 0.0000, 0.4198, 0.0000, 0.0000, 0.0000, 0.2784],
        [0.0000, 0.2945, 0.3884, 0.3171, 0.0000, 0.0000, 0.0000, 0.0000]])

In [92]:
tns=torch.randn((2,2,2))

In [ ]:
tns=torch.tensor([[[2,3],
                   [0,1]],

          [[5 , 5],
         [-10,-10]]])

In [110]:
torch.mean(tns,dim=-1,dtype=float)

tensor([[  2.5000,   0.5000],
        [  5.0000, -10.0000]], dtype=torch.float64)

In [ ]:
mhat_ouput=torch.randn(4,GPT_CONFIG_124M["emb_dim"])

In [153]:
cls=Router()
cls.forward(mhat_ouput)

(tensor([[0.3680, 0.0000, 0.0000, 0.0000, 0.3029, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.3291],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4266, 0.0000, 0.0000, 0.0000,
          0.0000, 0.3531, 0.2203],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1651, 0.2461,
          0.0000, 0.0000, 0.5888],
         [0.0000, 0.0000, 0.0000, 0.3158, 0.0000, 0.0000, 0.0000, 0.3951, 0.0000,
          0.0000, 0.0000, 0.2890]], grad_fn=<SoftmaxBackward0>),
 tensor([[ 0, 11,  4],
         [ 5, 10, 11],
         [11,  8,  7],
         [ 7,  3, 11]]))